In [14]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image, ImageDraw, ImageFont
import pytesseract
from datasets import load_dataset
import requests
from io import BytesIO
import torch

In [2]:
processor = LayoutLMv3Processor.from_pretrained("nielsr/layoutlmv3-finetuned-cord")
model = LayoutLMv3ForTokenClassification.from_pretrained("nielsr/layoutlmv3-finetuned-cord")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RobertaTokenizer'. 
The class this function is called from is 'LayoutLMv3TokenizerFast'.


In [9]:
# Set the device
device = "cuda" if torch.cuda.is_available() else "mps"
model.to(device)

LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [10]:
# Get Open Food Facts dataset
image_prefix_url = "https://prices.openfoodfacts.org/img/"
dataset = load_dataset("hf-internal-testing/example-documents", split="test")
dataset_ofp = load_dataset("openfoodfacts/open-prices")['prices']
dataset_ofp = dataset_ofp.filter(lambda x: x['proof_type'] == 'RECEIPT')
image_url = image_prefix_url+dataset_ofp[0]['proof_file_path']
response = requests.get(image_url)
response.raise_for_status()
image = Image.open(BytesIO(response.content))
# image = dataset[2]["image"]
image.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# Process the input
width, height = image.size

encoded_inputs = processor(images=image, return_tensors="pt", padding="max_length", truncation=True)
for k,v in encoded_inputs.items():
    encoded_inputs[k] = v.to(device)

In [41]:
# Perform inference
with torch.no_grad():
    try:
        outputs = model(**encoded_inputs)
        predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()

        predicted_labels = [model.config.id2label[prediction] for prediction in predictions]

        # Get words and boxes
        words = processor.tokenizer.convert_ids_to_tokens(encoded_inputs.input_ids[0])
        boxes = encoded_inputs.bbox[0].tolist()

        # Filter special tokens
        filtered_words = []
        filtered_boxes = []
        filtered_predictions = []
        for word, box, prediction in zip(words, boxes, predicted_labels):
            if word not in processor.tokenizer.all_special_tokens:
                filtered_words.append(word)
                filtered_boxes.append(box)
                filtered_predictions.append(prediction)

        # Visualization
        image_with_boxes = image.copy()
        draw = ImageDraw.Draw(image_with_boxes)

        # Fixed font size on default load
        font_size = 15  # Set your desired default font size

        try:
            font = ImageFont.load_default(size=font_size) # Load default font with size
        except Exception as e:
            print(f"Error loading default font: {e}")
            font = ImageFont.load_default() # Load default font without size if error
            print("Using default font without specified size")


        for word, box, label in zip(filtered_words, filtered_boxes, filtered_predictions):
            x0, y0, x1, y1 = box
            x0 = int(x0 * width / 1000)
            y0 = int(y0 * height / 1000)
            x1 = int(x1 * width / 1000)
            y1 = int(y1 * height / 1000)

            draw.rectangle([x0, y0, x1, y1], outline="red", width=2)

            # Position text above the box
            # bbox = draw.textbbox((0, 0), f"{word} ({label})", font=font)
            bbox = draw.textbbox((0, 0), f"{label}", font=font) # Only the label
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
            text_x = x0 + (x1 - x0 - text_width) // 2  # Center horizontally
            text_y = y0 - text_height - 2  # Position above, with a small gap

            # Check if text goes out of image bounds and adjust if necessary
            if text_y < 0:
                text_y = y0 + 2 # If the text is above the image, put it below

            # draw.text((text_x, text_y), f"{word} ({label})", fill="blue", font=font)
            draw.text((text_x, text_y), f"{label}", fill="blue", font=font)

        image_with_boxes.show()
        # image_with_boxes.save("output_image.jpg")  # Save the image if needed

    except Exception as e:
        print(f"Error during inference: {e}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [45]:
import torch
import torch.nn.functional as F
import json


def extract_structured_data(image_bytes, confidence_threshold=0.8):
    """
    Extracts structured data from a document image using LayoutLMv3.

    Args:
        image_bytes: Bytes of the image content.
        confidence_threshold: Confidence threshold for predictions.

    Returns:
        A JSON string representing the structured data, or None if an error occurs.
    """
    try:
        # Load image from bytes
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        width, height = image.size

        encoded_inputs = processor(images=image, return_tensors="pt", padding="max_length", truncation=True)
        for k,v in encoded_inputs.items():
            encoded_inputs[k] = v.to(device)

        with torch.no_grad():
            outputs = model(**encoded_inputs)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=-1)
            predicted_class_indices = logits.argmax(dim=-1).squeeze().tolist()
            predicted_probabilities = probabilities.max(dim=-1).values.squeeze().tolist()
            predicted_labels = [model.config.id2label[prediction] for prediction in predicted_class_indices]

            words = processor.tokenizer.convert_ids_to_tokens(encoded_inputs.input_ids[0])
            boxes = encoded_inputs.bbox[0].tolist()

            extracted_data = []
            for word, box, label, prob in zip(words, boxes, predicted_labels, predicted_probabilities):
                if word not in processor.tokenizer.all_special_tokens and prob >= confidence_threshold:
                    x0, y0, x1, y1 = box
                    x0 = int(x0 * width / 1000)
                    y0 = int(y0 * height / 1000)
                    x1 = int(x1 * width / 1000)
                    y1 = int(y1 * height / 1000)
                    extracted_data.append({
                        "text": word,
                        "label": label,
                        "confidence": float(prob),
                        "box": [x0, y0, x1, y1]
                    })
            return json.dumps(extracted_data, indent=4, ensure_ascii=False)

    except Exception as e:
        print(f"Error extracting data: {e}")
        return None

# Example usage (assuming you have the image bytes from the response)
image_bytes = response.content
json_data = extract_structured_data(image_bytes)

if json_data:
    print(json_data)
    # You can also load it as a Python object:
    data = json.loads(json_data)
    #Access the data
    for item in data:
        print(item["text"], item["label"])
else:
    print("Failed to extract data.")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/homebrew/Caskroom/miniforge/base/envs/ml-cli/lib/python3.12/site-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


[
    {
        "text": "ĠST",
        "label": "B-MENU.NM",
        "confidence": 0.9980173110961914,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "AL",
        "label": "I-MENU.NM",
        "confidence": 0.9959381818771362,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "ING",
        "label": "I-MENU.NM",
        "confidence": 0.9976460337638855,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "R",
        "label": "I-MENU.NM",
        "confidence": 0.9979369640350342,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "AD",
        "label": "I-MENU.NM",
        "confidence": 0.9979259967803955,
        "box": [
            738,
            342,
            1072,
            405
    

In [49]:
def extract_item_price_pairs(image_bytes, confidence_threshold=0.8):
    try:
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        width, height = image.size

        encoded_inputs = processor(images=image, return_tensors="pt", padding="max_length", truncation=True)
        for k,v in encoded_inputs.items():
            encoded_inputs[k] = v.to(device)

        with torch.no_grad():
            outputs = model(**encoded_inputs)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=-1)
            predicted_class_indices = logits.argmax(dim=-1).squeeze().tolist()
            predicted_probabilities = probabilities.max(dim=-1).values.squeeze().tolist()
            predicted_labels = [model.config.id2label[prediction] for prediction in predicted_class_indices]

            words = processor.tokenizer.convert_ids_to_tokens(encoded_inputs.input_ids[0])
            boxes = encoded_inputs.bbox[0].tolist()

            extracted_data = []
            for word, box, label, prob in zip(words, boxes, predicted_labels, predicted_probabilities):
                if word not in processor.tokenizer.all_special_tokens and prob >= confidence_threshold:
                    x0, y0, x1, y1 = box
                    x0 = int(x0 * width / 1000)
                    y0 = int(y0 * height / 1000)
                    x1 = int(x1 * width / 1000)
                    y1 = int(y1 * height / 1000)
                    extracted_data.append({
                        "text": word,
                        "label": label,
                        "confidence": float(prob),
                        "box": [x0, y0, x1, y1]
                    })
            print("All Extracted Data:") # Debugging
            print(json.dumps(extracted_data, indent=4, ensure_ascii=False))
            item_price_pairs = []
            items = [item for item in extracted_data if item["label"] == "ITEM"]
            amounts = [item for item in extracted_data if item["label"] == "AMOUNT"]

            # Simple proximity-based matching (improve as needed)
            for item in items:
                best_match = None
                min_distance = float('inf')
                for amount in amounts:
                    distance = ((item["box"][0] - amount["box"][0])**2 + (item["box"][1] - amount["box"][1])**2)**0.5 # Euclidean distance
                    if distance < min_distance and abs(item["box"][1]-amount["box"][1])<50: # added vertical distance threshold
                        min_distance = distance
                        best_match = amount
                if best_match:
                    item_price_pairs.append({
                        "item": item["text"],
                        "price": best_match["text"],
                        "item_box": item["box"],
                        "price_box": best_match["box"],
                        "item_confidence": item["confidence"],
                        "price_confidence": best_match["confidence"]
                    })


            return json.dumps(item_price_pairs, indent=4, ensure_ascii=False)

    except Exception as e:
        print(f"Error extracting data: {e}")
        return None

# Example usage (assuming you have the image bytes from the response)
image_bytes = response.content
json_data = extract_item_price_pairs(image_bytes)

if json_data:
    print(json_data)
    # You can also load it as a Python object:
    data = json.loads(json_data)
    #Access the data
    for item in data:
        print(item["item"], item["price"])
else:
    print("Failed to extract data.")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/homebrew/Caskroom/miniforge/base/envs/ml-cli/lib/python3.12/site-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


All Extracted Data:
[
    {
        "text": "ĠST",
        "label": "B-MENU.NM",
        "confidence": 0.9980173110961914,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "AL",
        "label": "I-MENU.NM",
        "confidence": 0.9959381818771362,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "ING",
        "label": "I-MENU.NM",
        "confidence": 0.9976460337638855,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "R",
        "label": "I-MENU.NM",
        "confidence": 0.9979369640350342,
        "box": [
            738,
            342,
            1072,
            405
        ]
    },
    {
        "text": "AD",
        "label": "I-MENU.NM",
        "confidence": 0.9979259967803955,
        "box": [
            738,
            342,
            1072,
